In [1]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')


Mounted at /content/drive


In [27]:
restaurant_path = '/content/drive/MyDrive/DS5500 Capstone/yelp_academic_dataset_business.json'
review_path = '/content/drive/MyDrive/DS5500 Capstone/yelp_academic_dataset_review.json'
user_path = '/content/drive/MyDrive/DS5500 Capstone/yelp_academic_dataset_user.json'

In [3]:
restaurant_df = pd.read_json(restaurant_path, lines = True)
review_df = pd.read_json(review_path, lines = True)
user_df = pd.read_json(user_path, lines = True)

In [4]:
restaurant_df.head()

In [5]:
my_state = 'CA'
restaurant_ids = set(restaurant_df[restaurant_df['state'] == my_state]['business_id'])

In [17]:
CA_Restaurants = restaurant_df[restaurant_df['state'] == my_state]

In [28]:
CA_Restaurants = CA_Restaurants.reset_index(drop=True)
CA_Restaurants.to_csv("/content/drive/MyDrive/DS5500 Capstone/CA_Restaurants.csv")

In [6]:
review_df.head()

In [7]:
user_ids = set(review_df[review_df['business_id'].isin(restaurant_ids)]['user_id'])

In [35]:
user_df[user_df['user_id'].isin(user_ids)].reset_index(drop=True).to_csv("/content/drive/MyDrive/DS5500 Capstone/CA_Users.csv")

In [8]:
#Rows are user ids
#Columns are restaurant ids

relevant_reviews_df = review_df[review_df['business_id'].isin(restaurant_ids)]


In [9]:
relevant_reviews_df.head()

In [32]:
relevant_reviews_df.reset_index(drop=True).to_csv("/content/drive/MyDrive/DS5500 Capstone/CA_Reviews.csv")

In [10]:
review_matrix = relevant_reviews_df.pivot_table(index='user_id', columns='business_id', values='stars', aggfunc='first', fill_value=0)

In [11]:
from scipy.sparse import csr_matrix

sparse_review_matrix = csr_matrix(review_matrix.values)

In [12]:
len(review_matrix.columns)

5203

In [13]:
!pip install implicit
from scipy.sparse import csr_matrix
from implicit import als
import numpy as np


# 2. Train ALS model
model = als.AlternatingLeastSquares(
    factors=50,        # latent dimensions
    regularization=0.1,
    iterations=20,
    use_gpu=True
)
model.fit(sparse_review_matrix)

# 3. Get recommendations for a user
user_idx = 0  # index into your matrix
ids, scores = model.recommend(
    user_idx,
    sparse_review_matrix[user_idx],
    N=10                    # top 10 recommendations
    #filter_already_liked=True
)

# Map indices back to restaurant ids
recommended = review_matrix.columns[ids]

In [14]:
list(recommended)

['iMTjejk6apJKzCukZnDw5A',
 'r2IhvKZQ_wLR5mLBnPOilg',
 'H-1qpp_77KggOAr9htUrEw',
 'l-rGtJt0E7PAklT0IK7oFQ',
 'S3QHy1sshUeZwXOYviVsXQ',
 'b5hxsRw76UUXcotxlCipuQ',
 'wFES5bGDiPANW13hNHnXOQ',
 'VOcGcN0bvGU_nzxbJgR5jQ',
 'cAbdvzqtFLaAAMFIyPf2AA',
 'Qkg16mN-8QR66gjzg6gMyw']

In [15]:
restaurant_df[restaurant_df['business_id'].isin(list(recommended))]

In [36]:
CA_Restaurants